# R2-MJ-2 — pair-level tables for the optimism-bias analysis

Referee 2, major comment 2 objects that the "PAV + genetic support in 2–5 therapeutic areas"
definition (OR = 10.3) was selected to maximise the odds ratio and never validated out of sample.

This notebook does **no** validation. It only builds the inputs the validation needs, and proves
they reproduce the published numbers exactly:

| Export | Rows | Contents |
| ------ | ---- | -------- |
| `chembl_ti_pairs_maxphase-r1.parquet` | ChEMBL T–I pairs | `targetId`, `diseaseId`, `maxClinicalPhase` (non-oncology, max phase > 0.5) |
| `l2g_indirect_assoc_all-r1.parquet` | target × disease | max L2G score propagated through the disease ontology, all credible sets |
| `l2g_indirect_assoc_pav-r1.parquet` | target × disease | the same, restricted to credible sets containing a protein-altering variant (`VEP == 1`) |
| `ti_pairs_chembl_master-r1.parquet` | ChEMBL T–I pairs | the three above joined, plus gene-level `uniqueTherapeuticAreas` / `uniqueDiseases` |
| `ti_pairs_pharmaprojects_master-r1.parquet` | Pharmaprojects T–I pairs | same columns for the Pharmaprojects (Minikel et al.) pairs |

Everything downstream (Phases 0–2) runs on these tables in pandas, with no Spark.

## Why a pair-level master table is sufficient

`chemblDrugEnrichment.drug_enrichemnt_from_evidence` ends in a **right join** of the indirect
association table onto the ChEMBL pairs, then a Fisher test on a 2×2 table. Two consequences:

1. The published enrichment is a function of the ChEMBL pairs alone — nothing at credible-set level
   survives the join except the maximum propagated score.
2. The therapeutic-area window is a **gene-level** filter (`g_p_s.uniqueTherapeuticAreas`), and
   ontology propagation is per target, so restricting the gene set before propagation is identical
   to filtering the joined table afterwards. This notebook asserts that equivalence numerically
   rather than assuming it.

The PAV filter is *not* gene-level — it selects credible sets — so it needs its own propagated
score column. Hence two indirect-association exports, not one.

In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/11 12:34:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
path_to_release_folder = "../../../data/25.06/"
path_to_intermediate_data_folder = "../../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_orig = session.spark.read.parquet(path_to_release_folder + "output/disease/disease.parquet")
chembl_evidence = session.spark.read.parquet(path_to_release_folder + "output/evidence/sourceId=chembl")

l2g_full = session.spark.read.parquet(path_to_intermediate_data_folder + "l2g_full_for_enrichment/")
g_p_s = session.spark.read.parquet(path_to_intermediate_data_folder + "genes_therapeutic_areas")

print("l2g rows:", l2g_full.count())
print("genes with pleiotropy annotation:", g_p_s.count())

l2g rows: 70400
genes with pleiotropy annotation: 8285


## ChEMBL target–indication pairs

Identical call to the published notebooks: all descendants of `MONDO_0045024` (cancer) removed,
max clinical phase per target–indication pair, pairs with phase > 0.5 kept.

In [4]:
efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids=["MONDO_0045024"],
)
print("oncology EFO ids removed:", len(efo_to_remove))

chembl = chemblDrugEnrichment.process_chembl_evidence(chembl_evidence, efo_to_remove).cache()
n_chembl = chembl.count()
print("ChEMBL T-I pairs:", n_chembl)
chembl.groupBy("maxClinicalPhase").count().orderBy("maxClinicalPhase").show()

oncology EFO ids removed: 3593


ChEMBL T-I pairs: 37377


+----------------+-----+
|maxClinicalPhase|count|
+----------------+-----+
|             1.0| 6163|
|             2.0|14410|
|             3.0|12240|
|             4.0| 4564|
+----------------+-----+



## Propagated (indirect) association scores

`min_score=0.1` at the evidence level and `use_max=True` in the propagation, exactly as published.
Because propagation takes the maximum over rows that already passed the 0.1 filter, thresholding
the propagated score at 0.1 afterwards is a no-op — every exported score is ≥ 0.1 by construction.
That is asserted below, so the downstream notebooks can treat "row present" as "genetic support".

In [5]:
def indirect_assoc(l2g_table):
    """Propagate L2G scores of one credible-set subset through the disease ontology."""
    evid = chemblDrugEnrichment.to_disease_target_evidence(
        table_with_score=l2g_table.drop("diseaseIds"),
        score_column="score",
        datasource_id="l2g",
        study_locus=sl,
        study_index=si,
        min_score=0.1,
    )
    n_evid = evid.count()
    ind = chemblDrugEnrichment.evidence_to_indirect_assosiations(
        evid,
        disease_index_orig,
        use_max=True,
        efo_to_remove=efo_to_remove,
    ).cache()
    return ind, n_evid


ind_all, n_evid_all = indirect_assoc(l2g_full)
print("evidence rows (all):", n_evid_all, "| indirect assoc (all):", ind_all.count())

ind_pav, n_evid_pav = indirect_assoc(l2g_full.filter(f.col("VEP") == 1))
print("evidence rows (PAV):", n_evid_pav, "| indirect assoc (PAV):", ind_pav.count())

evidence rows (all): 77071 | indirect assoc (all): 151704


evidence rows (PAV): 10275 | indirect assoc (PAV): 22509


In [6]:
# One row per target-disease pair, and every score above the 0.1 support threshold.
for name, ind in [("all", ind_all), ("pav", ind_pav)]:
    n = ind.count()
    n_distinct = ind.select("targetId", "diseaseId").distinct().count()
    n_below = ind.filter(f.col("indirect_assoc_score") < 0.1).count()
    print(f"{name}: rows={n} distinct pairs={n_distinct} scores below 0.1={n_below}")
    assert n == n_distinct, "indirect association table is not unique on (targetId, diseaseId)"
    assert n_below == 0, "propagated scores below the 0.1 support threshold exist"

assert g_p_s.count() == g_p_s.select("geneId").distinct().count(), "g_p_s is not unique on geneId"

all: rows=151704 distinct pairs=151704 scores below 0.1=0


pav: rows=22509 distinct pairs=22509 scores below 0.1=0


In [7]:
gene_features = g_p_s.select(
    f.col("geneId").alias("targetId"),
    "uniqueTherapeuticAreas",
    "uniqueDiseases",
    "approvedSymbol",
)

ind_all.write.mode("overwrite").parquet(path_to_intermediate_data_folder + "l2g_indirect_assoc_all-r1.parquet")
ind_pav.write.mode("overwrite").parquet(path_to_intermediate_data_folder + "l2g_indirect_assoc_pav-r1.parquet")
chembl.write.mode("overwrite").parquet(path_to_intermediate_data_folder + "chembl_ti_pairs_maxphase-r1.parquet")
print("exported")

exported


## The 2,734 gene–disease associations of the published strict definition

Supplementary Table st7 counts distinct gene–disease associations, not T–I pairs: the L2G table
filtered to `VEP == 1` and to genes with 2–5 therapeutic areas, exploded over `diseaseIds`.

In [8]:
window_genes = (
    g_p_s.filter(f.col("uniqueTherapeuticAreas") < 6)
    .filter(f.col("uniqueTherapeuticAreas") > 1)
    .select("geneId")
    .distinct()
)
print("genes in the 2-5 TA window:", window_genes.count())

strict_assoc = (
    l2g_full.filter(f.col("VEP") == 1)
    .join(window_genes, on="geneId", how="inner")
    .withColumn("diseaseId", f.explode("diseaseIds"))
    .select("geneId", "diseaseId")
    .distinct()
)
n_strict_assoc = strict_assoc.count()
print("strict-definition gene-disease associations:", n_strict_assoc, "(published: 2734)")
assert n_strict_assoc == 2734

genes in the 2-5 TA window: 4028


strict-definition gene-disease associations: 2734 (published: 2734)


## Master table — ChEMBL

Left joins onto the ChEMBL pairs. `score_all` / `score_pav` are null where the target–indication
pair has no propagated genetic support of that kind. Gene-level pleiotropy is null for targets
absent from `genes_therapeutic_areas` (i.e. targets with no disease GWAS association at all);
`in_gps` records that, because the published window filter was an inner join and therefore dropped
those targets.

In [9]:
import pandas as pd

master_sdf = (
    chembl.join(
        ind_all.withColumnRenamed("indirect_assoc_score", "score_all"),
        ["targetId", "diseaseId"],
        "left",
    )
    .join(
        ind_pav.withColumnRenamed("indirect_assoc_score", "score_pav"),
        ["targetId", "diseaseId"],
        "left",
    )
    .join(gene_features, "targetId", "left")
)

master = master_sdf.toPandas()
print(master.shape)
assert len(master) == n_chembl, "joins changed the number of ChEMBL pairs"

master["in_gps"] = master["uniqueTherapeuticAreas"].notna()
master["approved"] = (master["maxClinicalPhase"] >= 4).astype(int)
master.head()

(37377, 8)


,targetId,diseaseId,maxClinicalPhase,score_all,score_pav,uniqueTherapeuticAreas,uniqueDiseases,approvedSymbol,in_gps,approved
0,ENSG00000007314,EFO_0000555,2.0,NaN,NaN,NaN,NaN,None,False,0
1,ENSG00000007314,EFO_0004699,3.0,NaN,NaN,NaN,NaN,None,False,0
2,ENSG00000007314,EFO_0801084,2.0,NaN,NaN,NaN,NaN,None,False,0
3,ENSG00000010310,EFO_0003884,2.0,NaN,NaN,9.0,15.0,GIPR,True,0
4,ENSG00000012504,MONDO_0019052,4.0,NaN,NaN,NaN,NaN,None,False,1


In [10]:
print("pairs with any genetic support:", master["score_all"].notna().sum())
print("pairs with PAV genetic support:", master["score_pav"].notna().sum())
print("pairs whose target has no pleiotropy annotation:", (~master["in_gps"]).sum())
print("approved (phase 4) pairs:", master["approved"].sum())
print()
print(master.groupby("maxClinicalPhase").size())

pairs with any genetic support: 742
pairs with PAV genetic support: 161
pairs whose target has no pleiotropy annotation: 18897
approved (phase 4) pairs: 4564

maxClinicalPhase
1.0     6163
2.0    14410
3.0    12240
4.0     4564
dtype: int64


## Enrichment statistics, reimplemented

`or_rs` reproduces `chemblDrugEnrichment.drug_enrichemnt_from_evidence` exactly: Fisher's exact
odds ratio, relative success as a risk ratio, and Woolf/log CIs with z = 1.96. `support` is the
boolean genetic-support column, `approved` the phase-4 outcome.

In [11]:
import numpy as np
from scipy.stats import chi2, fisher_exact


def or_rs(support, approved, phase_label="4+"):
    """OR and relative success for one genetic-support definition.

    Args:
        support: boolean array, genetic support present for the T-I pair
        approved: 0/1 array, pair reached the phase of interest
        phase_label: label carried into the output row

    Returns:
        dict of the same fields the published enrichment table reports
    """
    support = np.asarray(support, dtype=bool)
    approved = np.asarray(approved, dtype=int)

    N_G = int(support.sum())
    N_negG = int((~support).sum())
    X_G = int(approved[support].sum())
    X_negG = int(approved[~support].sum())

    table = [[N_negG - X_negG, X_negG], [N_G - X_G, X_G]]
    odds_ratio, p_value = fisher_exact(table)

    if np.any(np.array(table) == 0):
        return {
            "clinicalPhase": phase_label,
            "odds_ratio": 1.0,
            "p_value": p_value,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "relative_success": 1.0,
            "ci_rs_low": np.nan,
            "ci_rs_high": np.nan,
            "rs_p_value": 1.0,
            "no_evid-low_clinphase": table[0][0],
            "no_evid-high_clinphase": X_negG,
            "yes_evid-low_clinphase": table[1][0],
            "yes_evid-high_clinphase": X_G,
        }

    z = 1.96
    ln_or = np.log(odds_ratio)
    se_ln_or = np.sqrt(1 / table[0][0] + 1 / table[0][1] + 1 / table[1][0] + 1 / table[1][1])

    rs = (X_G / N_G) / (X_negG / N_negG)
    ln_rs = np.log(rs)
    se_ln_rs = np.sqrt((1 / X_negG) - (1 / N_negG) + (1 / X_G) - (1 / N_G))

    return {
        "clinicalPhase": phase_label,
        "odds_ratio": float(odds_ratio),
        "p_value": float(p_value),
        "ci_low": float(np.exp(ln_or - z * se_ln_or)),
        "ci_high": float(np.exp(ln_or + z * se_ln_or)),
        "relative_success": float(rs),
        "ci_rs_low": float(np.exp(ln_rs - z * se_ln_rs)),
        "ci_rs_high": float(np.exp(ln_rs + z * se_ln_rs)),
        "rs_p_value": float(chi2.sf((ln_rs / se_ln_rs) ** 2, df=1)),
        "no_evid-low_clinphase": table[0][0],
        "no_evid-high_clinphase": X_negG,
        "yes_evid-low_clinphase": table[1][0],
        "yes_evid-high_clinphase": X_G,
    }


def support_mask(df, pav=False, ta_min=None, ta_max=None):
    """Genetic-support mask for one (PAV, TA window) definition.

    ta_min / ta_max are inclusive; None means unbounded. A TA window implies the target must be
    present in genes_therapeutic_areas, matching the published inner join.
    """
    mask = df["score_pav"].notna() if pav else df["score_all"].notna()
    if ta_min is not None or ta_max is not None:
        mask = mask & df["in_gps"]
        ta = df["uniqueTherapeuticAreas"]
        if ta_min is not None:
            mask = mask & (ta >= ta_min)
        if ta_max is not None:
            mask = mask & (ta <= ta_max)
    return mask

## Reproduction of the published numbers

Target values from the published notebooks (`06-target-enrichment/02-enrichment-groups.ipynb`
and `09-best_category_description.ipynb`):

| Definition | OR | RS | approved supported pairs |
| ---------- | -- | -- | ------------------------ |
| all GWAS | 3.618578 | 2.755 | 242 |
| PAV + 2–5 TA | 10.288962 | 4.843708 | 51 |

**The count is 51, not 52.** The reviewer-response brief and the manuscript text say 52 of 242
(21.5%). The published enrichment table in `09-best_category_description.ipynb` reports
`yes_evid-high_clinphase = 51`, with the 2×2 table [[32777, 4513], [36, 51]] — and that table is
what produced the published OR: 32777 × 51 / (4513 × 36) = 10.2890, whereas 52 would give 10.4907.
The four cells also sum to 37,377, the published number of T–I pairs. So 51 is the value consistent
with every other published number, and the correct share is 51/242 = 21.1%. Flagged for the
response letter; it is not introduced by anything in this notebook.

In [12]:
checks = pd.DataFrame(
    [
        {"definition": "all GWAS", **or_rs(support_mask(master), master["approved"])},
        {
            "definition": "PAV + 2-5 TA",
            **or_rs(support_mask(master, pav=True, ta_min=2, ta_max=5), master["approved"]),
        },
    ]
)
checks.set_index("definition").T

definition,all GWAS,PAV + 2-5 TA
clinicalPhase,4+,4+
odds_ratio,3.618578,10.288962
p_value,0.0,0.0
ci_low,3.093638,6.707865
ci_high,4.232591,15.781881
relative_success,2.76454,4.843708
ci_rs_low,2.483639,4.051253
ci_rs_high,3.077211,5.791173
rs_p_value,0.0,0.0
no_evid-low_clinphase,32313,32777


In [13]:
all_gwas = checks.loc[checks["definition"] == "all GWAS"].iloc[0]
strict = checks.loc[checks["definition"] == "PAV + 2-5 TA"].iloc[0]

assert np.isclose(all_gwas["odds_ratio"], 3.618578, rtol=1e-5), all_gwas["odds_ratio"]
assert all_gwas["yes_evid-high_clinphase"] == 242, all_gwas["yes_evid-high_clinphase"]
assert np.isclose(strict["odds_ratio"], 10.288962, rtol=1e-5), strict["odds_ratio"]
assert np.isclose(strict["relative_success"], 4.843708, rtol=1e-5), strict["relative_success"]
assert strict["yes_evid-high_clinphase"] == 51, strict["yes_evid-high_clinphase"]
assert strict["no_evid-low_clinphase"] == 32777
assert strict["no_evid-high_clinphase"] == 4513
assert strict["yes_evid-low_clinphase"] == 36
assert (
    strict["no_evid-low_clinphase"]
    + strict["no_evid-high_clinphase"]
    + strict["yes_evid-low_clinphase"]
    + strict["yes_evid-high_clinphase"]
) == 37377
print("published OR = 10.3 / RS = 4.8 / 2x2 [[32777, 4513], [36, 51]] reproduced from the pair-level table")
print(
    f"{strict['yes_evid-high_clinphase']} of {all_gwas['yes_evid-high_clinphase']} approved "
    f"GWAS-supported pairs match the strict definition "
    f"({100 * strict['yes_evid-high_clinphase'] / all_gwas['yes_evid-high_clinphase']:.1f}%)"
)

published OR = 10.3 / RS = 4.8 / 2x2 [[32777, 4513], [36, 51]] reproduced from the pair-level table
51 of 242 approved GWAS-supported pairs match the strict definition (21.1%)


## Master table — Pharmaprojects

`minikel_etal_processed_data_v2.csv` is the processed Pharmaprojects table from
`chapters/05-other-drug-indication-data/01-process-minikel_etal_data.ipynb`: gene symbols mapped to
Ensembl ids, MeSH indications mapped to EFO/MONDO through the disease index, oncology removed,
restricted to `ccatnum >= 2` (phase I or beyond). `ccatnum` runs 2–5 for phase I–launched, so
`maxClinicalPhase = ccatnum - 1` and `outcome = (ccatnum == 5)`.

`geneticSupport_old` is **their** genetic-support call (`target_status == "genetically supported
target"`), which is what the external self-validation below uses. The PAV and score columns are
re-derived here from our own propagation so the frozen definition can be evaluated on their pairs.

In [14]:
pp = pd.read_csv(
    path_to_intermediate_data_folder + "minikel_etal_processed_data_v2.csv",
    low_memory=False,
)
print(pp.shape)
print(pp.groupby("ccatnum").size())

keep = [
    "targetId",
    "diseaseId",
    "meshId",
    "approvedSymbol",
    "ti_uid",
    "ccatnum",
    "maxClinicalPhase",
    "outcome",
    "geneticSupport_old",
    "genetic_insight",
    "target_status",
    "year_launch",
    "orphan",
]
pp = pp[keep].copy()
assert pp["maxClinicalPhase"].equals(pp["ccatnum"].astype(int) - 1)
assert pp["outcome"].equals((pp["ccatnum"] == 5).astype(int))

(7390, 48)
ccatnum
2    2274
3    3303
4     900
5     913
dtype: int64


In [15]:
ind_all_pd = ind_all.toPandas().rename(columns={"indirect_assoc_score": "score_all"})
ind_pav_pd = ind_pav.toPandas().rename(columns={"indirect_assoc_score": "score_pav"})
gene_features_pd = gene_features.toPandas()

n_before = len(pp)
pp_master = (
    pp.merge(ind_all_pd, on=["targetId", "diseaseId"], how="left")
    .merge(ind_pav_pd, on=["targetId", "diseaseId"], how="left")
    .merge(gene_features_pd.drop(columns=["approvedSymbol"]), on="targetId", how="left")
)
assert len(pp_master) == n_before, "joins changed the number of Pharmaprojects pairs"

pp_master["in_gps"] = pp_master["uniqueTherapeuticAreas"].notna()
pp_master["approved"] = pp_master["outcome"].astype(int)
print(pp_master.shape)
print("pairs with any genetic support:", pp_master["score_all"].notna().sum())
print("pairs with PAV genetic support:", pp_master["score_pav"].notna().sum())
print("launched pairs:", int(pp_master["approved"].sum()))

(7390, 19)
pairs with any genetic support: 469
pairs with PAV genetic support: 137
launched pairs: 913


### Self-validation of the Pharmaprojects table

Two checks before this table is used for anything:

1. **Their evidence, their outcome.** Regressing launch on Pharmaprojects' own genetic-support flag
   must recover the published effect (`01-process-minikel_etal_data.ipynb`: coefficient 0.8431,
   OR = 2.32). Nelson 2015 and Minikel 2024 report roughly a doubling of success, so an OR near 2
   is the expected value; a null here would mean the table is mis-joined.
2. **Our evidence, their outcome.** The all-GWAS enrichment on Pharmaprojects pairs, for
   comparison against the ChEMBL OR of 3.62.

In [16]:
import statsmodels.formula.api as smf

m_own = smf.logit("approved ~ geneticSupport_old", data=pp_master).fit(disp=False)
or_own = float(np.exp(m_own.params["geneticSupport_old"]))
ci_own = np.exp(m_own.conf_int().loc["geneticSupport_old"].values)
print(
    f"their flag:  OR = {or_own:.3f} [{ci_own[0]:.3f}, {ci_own[1]:.3f}] "
    f"p = {m_own.pvalues['geneticSupport_old']:.3g} coef = {m_own.params['geneticSupport_old']:.4f}"
)
assert np.isclose(m_own.params["geneticSupport_old"], 0.8431, atol=5e-4), "Minikel self-check failed"

own_fisher = or_rs(pp_master["geneticSupport_old"].astype(bool), pp_master["approved"])
pp_checks = pd.DataFrame(
    [
        {"definition": "Pharmaprojects own flag", **own_fisher},
        {"definition": "our all-GWAS support", **or_rs(support_mask(pp_master), pp_master["approved"])},
    ]
)
pp_checks.set_index("definition").T

their flag:  OR = 2.323 [1.944, 2.777] p = 1.84e-20 coef = 0.8431


definition,Pharmaprojects own flag,our all-GWAS support
clinicalPhase,4+,4+
odds_ratio,2.3235,1.654614
p_value,0.0,0.000112
ci_low,1.944189,1.295204
ci_high,2.776813,2.113757
relative_success,2.027332,1.534578
ci_rs_low,1.758295,1.254627
ci_rs_high,2.337535,1.876996
rs_p_value,0.0,0.000031
no_evid-low_clinphase,5811,6094


## Overlap between the two resources

The Pharmaprojects launched set and the ChEMBL phase-4 set describe the same approved drugs, so
Phase 2 cannot be called replication in independent data. This quantifies how much of it is shared
before any result is labelled external.

In [17]:
chembl_pairs = set(map(tuple, master.loc[master["approved"] == 1, ["targetId", "diseaseId"]].values))
pp_pairs = set(map(tuple, pp_master.loc[pp_master["approved"] == 1, ["targetId", "diseaseId"]].values))
shared = chembl_pairs & pp_pairs

chembl_all = set(map(tuple, master[["targetId", "diseaseId"]].values))
pp_all = set(map(tuple, pp_master[["targetId", "diseaseId"]].values))

overlap = pd.DataFrame(
    [
        {
            "set": "approved / launched T-I pairs",
            "chembl": len(chembl_pairs),
            "pharmaprojects": len(pp_pairs),
            "shared": len(shared),
            "shared_pct_of_pharmaprojects": 100 * len(shared) / len(pp_pairs),
        },
        {
            "set": "all T-I pairs",
            "chembl": len(chembl_all),
            "pharmaprojects": len(pp_all),
            "shared": len(chembl_all & pp_all),
            "shared_pct_of_pharmaprojects": 100 * len(chembl_all & pp_all) / len(pp_all),
        },
    ]
)
overlap

,set,chembl,pharmaprojects,shared,shared_pct_of_pharmaprojects
0,approved / launched T-I pairs,4564,911,447,49.066959
1,all T-I pairs,37377,7367,2592,35.183928


## Export

In [18]:
master.to_parquet(path_to_intermediate_data_folder + "ti_pairs_chembl_master-r1.parquet", index=False)
pp_master.to_parquet(path_to_intermediate_data_folder + "ti_pairs_pharmaprojects_master-r1.parquet", index=False)
checks.to_csv("reproduction_checks-r1.csv", index=False)
pp_checks.to_csv("pharmaprojects_checks-r1.csv", index=False)
overlap.to_csv("chembl_pharmaprojects_overlap-r1.csv", index=False)

print("ChEMBL master:", master.shape)
print("Pharmaprojects master:", pp_master.shape)
print(sorted(master.columns))

ChEMBL master: (37377, 10)
Pharmaprojects master: (7390, 19)
['approved', 'approvedSymbol', 'diseaseId', 'in_gps', 'maxClinicalPhase', 'score_all', 'score_pav', 'targetId', 'uniqueDiseases', 'uniqueTherapeuticAreas']
